In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

from src.codebookmap import BRFSS_MAP
from src.armpipelines import make_arm_a
from src.armpipelines import make_arm_b

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.codebook_map import BRFSS_MAP

In [ ]:
dev = pd.read_parquet("../../data/dev.parquet")
df_arm_b = make_arm_b(dev)
df_arm_a_X, df_arm_a_y = make_arm_a(dev)

In [ ]:
# Group column names based on BRFSS dictionary
nominal_cols = [
    col for col, rules in BRFSS_MAP.items() 
    if rules['type'] == 'nominal' and not rules.get('is_target')
]
numeric_cols = [
    col for col, rules in BRFSS_MAP.items() 
    if rules['type'] in ['ordinal', 'continuous'] and not rules.get('is_target')
]


In [ ]:
# Numeric branch (Arm A)
naive_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean', add_indicator=False)),
    ('scaler', StandardScaler())
])

# Nominal branch (Arm A)
naive_nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# preprocessor (Arm A)
preprocessor_arm_a = ColumnTransformer(
    transformers=[
        ('num', naive_numeric_transformer, numeric_cols),
        ('cat', naive_nominal_transformer, nominal_cols)
    ]
)

In [ ]:
#  Rule 1(Arm B) : One-Hot Encode (including 'Unknown')
nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

#  Rule 2 (Arm B): Impute -> Missing Indicators -> Standardize
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

# preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', nominal_transformer, nominal_cols)
    ]
)